# GRID grain size from photos — final solution (Kaggle-ready)

Companion code to **Competition_Story_Report.docx**. It runs the exact architecture from the story — the
v1 texture model (Public LB 36.19, per-point-k version 34.32) **+ Patch A (Delesse k = 1)** **+ Patch B (frozen DINOv2, gated)**.

| block | what it does | status in this notebook |
|---|---|---|
| **A. texture model** | train-like degradation → 4.5 px/mm → 5 DoG band energies → Ridge → Weibull (ln b, c) | always on |
| **B. stone measurement** | SAM on full-res photos → A63 / A20 → redraw the 63 mm / 20 mm points | always on (auto-skips if SAM cannot load) |
| **Patch A** | k = 1.0 by Delesse's stereological principle instead of the H374-calibrated k ≈ 1.22 | `CFG["k_mode"] = "delesse"`, all modes ablated in the stones section |
| **Patch B** | frozen DINOv2 embeddings (dino_rgb + dino_gray) concatenated to the 5 DoG features, same Ridge→Weibull head | **gated**: switched on only if cross-phone CV improves by ≥ 1.0 EMD |

**Kaggle settings:** Internet **ON** (SAM `facebook/sam-vit-base` + DINOv2 weights download on first run), Accelerator **GPU (T4)** — CPU also works (SAM ~25 min on CPU). Everything else is preinstalled on Kaggle.

**Outputs in `/kaggle/working`:** `submission.csv` (selected configuration), `submission_model_A_only.csv`, and `submission_k_delesse.csv` / `submission_k_v3.csv` / `submission_k_v4.csv` so each k-variant can be submitted separately (one change per submission).

**Reference numbers from the story report:** site × cross-phone CV 50.67, site-LOSO 47.53 for the 5-feature model; Public LB 36.19 (v4 k) / 34.32 (v3 per-point k); Patch A expected ≈ 34.6; Patch B only if the gate passes (29–33 hoped).

In [ ]:
import importlib.util
import io
import re
import socket
import time
import unicodedata
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

IS_KAGGLE = Path("/kaggle/input").exists()
if IS_KAGGLE:
    _hits = sorted(Path("/kaggle/input").glob("**/Training_labels*.csv"))
    if not _hits:
        raise FileNotFoundError("competition CSVs not found under /kaggle/input - attach the competition dataset")
    INPUT_DIR = _hits[0].parent
    OUTPUT_DIR = Path("/kaggle/working")
else:
    INPUT_DIR = Path("input")
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HAVE_TORCH = importlib.util.find_spec("torch") is not None
HAVE_TRANSFORMERS = importlib.util.find_spec("transformers") is not None
try:
    socket.create_connection(("pypi.org", 443), timeout=4)
    INTERNET = True
except OSError:
    INTERNET = False
if HAVE_TORCH:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    DEVICE = "cpu"

print("input        :", INPUT_DIR)
print("torch        :", HAVE_TORCH, "| transformers:", HAVE_TRANSFORMERS)
print("internet     :", INTERNET, "| device:", DEVICE)

In [ ]:
# ---- sizes, metric weights, config -----------------------------------------------------------
SIZES = [0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2, 6.3, 20, 63, 200]
SIZE_COLS = ["0.002", "0.0063", "0.02", "0.063", "0.2", "0.63", "2", "6.3", "20", "63", "200"]
W_EMD = np.diff(np.log10(SIZES))          # widths of the 10 log-size intervals (~0.5 each)
CAM_EDGE, CAM_A52 = "Motorola Edge", "Samsung A52"

CFG = {
    # ---- v1 texture model (unchanged from the story report, sections 6-7) ----
    "target_ppm": 4.5,                    # common physical scale [px/mm]
    "crop_frac": 0.5,                     # centre crop (avoid tray edges and vignetting)
    "dog_sigmas_mm": [0.25, 0.5, 1.0, 2.0, 4.0, 8.0],
    "train_like_above_ppm": 6.0,          # photos sharper than this go through the train-like path
    "ridge_alpha": 1.0,
    "weights_group": {"gravel": 0.2, "non_gravel": 0.8},   # CV weighting, resembles the test composition
    "grid_lnb": [-6.0, 4.0, 41], "grid_c": [0.1, 2.0, 39], "c_floor": 0.02,
    "monotone": "running_max",            # v1 exact; "l1_pava" = L1 isotonic projection (median PAVA)
    # ---- SAM stone measurement (v1, sections 9-10; fixed before looking at the LB) ----
    "sam_model": "facebook/sam-vit-base", "sam_min_ppm": 12.0, "work_ppm": 2.0, "work_crop": 0.12,
    "pred_iou_thresh": 0.86, "stability_score_thresh": 0.9, "points_per_batch": 64,
    "overlap_drop": 0.5, "min_fill": 0.35, "min_side_mm": 8.0, "max_area_frac": 0.5,
    "apply_if_A63_ge": 0.02,
    # ---- Patch A: the k factor that turns stone area share into mass share ----
    "k_mode": "delesse",                  # "delesse" (k=1, theory) | "v3" (k20=1.00, k63=0.92) | "v4" (H374-calibrated)
    # ---- Patch B: frozen DINOv2, gated on cross-phone CV ----
    "dino_model": "dinov2_vits14",
    "dino_px": 518,
    "dino_pca_dim": 24,                   # = number of training samples
    "gate_margin": 1.0,                   # cross-phone EMD advantage required to switch DINO on
}

# ---- file names are messy: parse camera + sample from the stem -------------------------------
CAMERA_RE = re.compile(r"^(Motorola_Edge_60_fusion|Motorola_Edge|Samsung_A52|iPhone_?14|iPhone_?16)_(.+?)(?:_\d+| \(\d+\))?$")
CAMERA_KEY = {"Motorola_Edge_60_fusion": "Motorola Edge 60 Fusion", "Motorola_Edge": CAM_EDGE, "Samsung_A52": CAM_A52,
              "iPhone14": "iPhone 14", "iPhone_14": "iPhone 14", "iPhone16": "iPhone 16", "iPhone_16": "iPhone 16"}


def normalize_id(raw: str) -> str:
    """Map the sample part of a file name to the submission ID spelling (ue/oe/ae, ',' to '_')."""
    s = unicodedata.normalize("NFC", raw).replace("ü", "ue").replace("ö", "oe").replace("ä", "ae")
    return s.replace(",", "_")


labels = pd.read_csv(INPUT_DIR / "Training_labels_updated.csv", dtype={"sample_id": str}).set_index("sample_id")
sample_sub = pd.read_csv(INPUT_DIR / "sample_submission.csv", dtype={"sample_id": str})
ppm_tab = pd.read_csv(INPUT_DIR / "ppm_updated.csv").set_index("phone")

rows = []
for split, pattern in [("train", "Training-All_Photos*"), ("test", "Test_All_Photos*")]:
    for p in sorted(INPUT_DIR.glob(f"{pattern}/**/*")):
        if p.suffix.lower() not in {".jpg", ".jpeg"}:
            continue
        m = CAMERA_RE.match(unicodedata.normalize("NFC", p.stem))
        if m is None:
            raise ValueError(f"unexpected file name: {p.name}")
        with Image.open(p) as im:
            w, h = im.size
        rows.append({"split": split, "path": p, "camera": CAMERA_KEY[m.group(1)],
                     "sample_id": normalize_id(m.group(2)), "w": w, "h": h})
photos = pd.DataFrame(rows)
photos["ppm_eff"] = [ppm_tab.loc[c, "ppm"] * max(w, h) / ppm_tab.loc[c, "width"]
                     for c, w, h in zip(photos.camera, photos.w, photos.h)]
assert sorted(set(photos.query("split == 'train'").sample_id)) == sorted(labels.index), "train photo/sample mismatch"
assert set(photos.query("split == 'test'").sample_id) == set(sample_sub.sample_id), "test photo/sample mismatch"

# soil type from the label (used only for CV weighting): fine / sand / gravel
lab = labels.copy()
lab["type"] = np.select([lab["0.063"] >= 40, (lab["63"] - lab["2"]) >= 40], ["fine", "gravel"], "sand")
lab["group"] = np.where(lab["type"] == "gravel", "gravel", "non_gravel")
print(photos.groupby(["split", "camera"]).agg(photos=("path", "size"), samples=("sample_id", "nunique"),
                                              px_per_mm=("ppm_eff", "median")).round(1))

In [ ]:
# ---- features: train-like degradation -> physical scale -> DoG band energies -------------------
_train_small = photos[(photos.split == "train") & (photos.ppm_eff <= CFG["train_like_above_ppm"])]
TRAIN_LIKE_PPM = float(_train_small.ppm_eff.median()) if len(_train_small) else CFG["target_ppm"]
# copy the JPEG quantisation tables from a real training photo instead of guessing a quality number
_qsrc = _train_small[_train_small.camera == CAM_A52]
if len(_qsrc) == 0:
    _qsrc = _train_small
TRAIN_QTABLES = Image.open(sorted(_qsrc.path)[0]).quantization
print(f"train-like target: {TRAIN_LIKE_PPM:.3f} px/mm")


def to_train_like(rgb: np.ndarray, ppm_eff: float) -> tuple:
    """Downscale a sharp photo to the training scale and re-save it as JPEG with the training q-tables (4:2:0)."""
    f = TRAIN_LIKE_PPM / ppm_eff
    small = cv2.resize(rgb, (round(rgb.shape[1] * f), round(rgb.shape[0] * f)), interpolation=cv2.INTER_AREA)
    buf = io.BytesIO()
    Image.fromarray(small).save(buf, format="JPEG", qtables=TRAIN_QTABLES, subsampling=2)
    return np.asarray(Image.open(io.BytesIO(buf.getvalue())).convert("RGB")), TRAIN_LIKE_PPM


def load_gray(path: Path, ppm_eff: float) -> np.ndarray:
    """EXIF rotation -> (train-like path if sharp) -> grayscale -> landscape -> 4.5 px/mm -> centre crop."""
    im = ImageOps.exif_transpose(Image.open(path))
    if ppm_eff > CFG["train_like_above_ppm"]:
        rgb, ppm_eff = to_train_like(np.asarray(im.convert("RGB")), ppm_eff)
        im = Image.fromarray(rgb)
    a = np.asarray(im.convert("L"))
    if a.shape[0] > a.shape[1]:
        a = np.ascontiguousarray(np.rot90(a))          # top-down shots: orientation carries no information
    f = CFG["target_ppm"] / ppm_eff
    a = cv2.resize(a, (round(a.shape[1] * f), round(a.shape[0] * f)), interpolation=cv2.INTER_AREA)
    h, w = a.shape
    ch, cw = int(h * CFG["crop_frac"]), int(w * CFG["crop_frac"])
    return a[(h - ch) // 2:(h - ch) // 2 + ch, (w - cw) // 2:(w - cw) // 2 + cw].astype(np.float32)


def dog_features(g: np.ndarray) -> dict:
    """Energy of the difference between neighbouring blur levels, relative to the image contrast (log).

    A 'sieve in image space': each of the 5 bands measures how much contrast lives at grains of that size.
    Dividing by the image std removes exposure/contrast differences; colour was dropped (systematic between phones)."""
    blurs = [cv2.GaussianBlur(g, (0, 0), s * CFG["target_ppm"]) for s in CFG["dog_sigmas_mm"]]
    sd = float(g.std()) + 1e-6
    return {f"dog_{s}mm": float(np.log(np.std(blurs[k - 1] - blurs[k]) / sd + 1e-6))
            for k, s in enumerate(CFG["dog_sigmas_mm"]) if k > 0}


t0 = time.time()
feats = pd.DataFrame([dog_features(load_gray(p, e)) for p, e in zip(photos.path, photos.ppm_eff)], index=photos.index)
FEATS = list(feats.columns)
photos = photos.join(feats)
print(f"{len(photos)} photos in {time.time() - t0:.0f} s; features: {FEATS}")

In [ ]:
# ---- competition metric, monotone projection, Weibull parameterisation ------------------------
_LOG_SIZES = np.log(np.asarray(SIZES, dtype=float))


def emd(y_true, y_pred):
    """Competition metric per sample: sum over the first 10 points of |error| x log10 interval width.

    On the half-log-decade sieve grid every weight is ~0.5, so EMD ~ 0.5 x sum|dF|: an L1 metric -
    medians and L1 losses are the right tools, and a 1-point error at every size costs ~5."""
    return (np.abs(np.asarray(y_true)[..., :10] - np.asarray(y_pred)[..., :10]) * W_EMD).sum(-1)


def l1_pava(y: np.ndarray) -> np.ndarray:
    """L1 isotonic projection (pool-adjacent-violators with weighted medians).

    The L1-optimal monotone approximation of a vector; used when CFG['monotone'] == 'l1_pava'."""
    y = np.asarray(y, dtype=float)
    stack = []
    for v in y:
        stack.append([v])
        while len(stack) >= 2 and np.median(stack[-2]) > np.median(stack[-1]):
            a, b = stack[-2], stack[-1]
            stack[-2:] = [a + b]
    return np.concatenate([np.full(len(b), np.median(b)) for b in stack])


def postprocess(curve) -> np.ndarray:
    """Valid submission row: clip to 0-100, make monotone (running max, or L1-PAVA), 200 mm = 100."""
    c = np.clip(np.asarray(curve, dtype=float), 0, 100)
    if CFG["monotone"] == "l1_pava":
        c = np.vstack([l1_pava(row) for row in c]) if c.ndim > 1 else l1_pava(c)
    else:
        c = np.maximum.accumulate(c, axis=-1)
    c[..., -1] = 100.0
    return c


def aggregate(per_img: pd.DataFrame) -> pd.DataFrame:
    """Average the per-photo curves of each sample, then post-process once."""
    agg = per_img.groupby(level=0).mean()
    return pd.DataFrame(postprocess(agg.to_numpy()), index=agg.index, columns=SIZE_COLS)


def weibull_curve(lnb, c) -> np.ndarray:
    """(ln b, c) -> % passing at the 11 sizes. c is floored to keep the curve defined."""
    lnb = np.asarray(lnb, dtype=float)[..., None]
    c = np.maximum(np.asarray(c, dtype=float), CFG["c_floor"])[..., None]
    return 100.0 * (1.0 - np.exp(-np.exp(c * (_LOG_SIZES - lnb))))


def fit_weibull_label(y: np.ndarray) -> tuple:
    """Fit (ln b, c) to one true curve by minimising the EMD itself (grid start -> Nelder-Mead).

    Minimising the competition metric (not least squares) keeps the fine end properly weighted."""
    g_lnb = np.linspace(CFG["grid_lnb"][0], CFG["grid_lnb"][1], int(CFG["grid_lnb"][2]))
    g_c = np.linspace(CFG["grid_c"][0], CFG["grid_c"][1], int(CFG["grid_c"][2]))
    L, C = np.meshgrid(g_lnb, g_c, indexing="ij")
    e = emd(y, weibull_curve(L.ravel(), C.ravel())).reshape(L.shape)
    k = np.unravel_index(int(np.argmin(e)), e.shape)
    x0 = np.array([L[k], C[k]])
    res = minimize(lambda x: float(emd(y, weibull_curve(x[0], x[1]))), x0, method="Nelder-Mead",
                   options={"xatol": 1e-4, "fatol": 1e-6, "maxiter": 2000})
    x = res.x if res.fun <= e[k] else x0
    return float(x[0]), float(max(x[1], CFG["c_floor"]))


# label-only transform: computed once per training sample (does not use any validation information)
TARGETS = pd.DataFrame({s: fit_weibull_label(lab.loc[s, SIZE_COLS].to_numpy(dtype=float)) for s in lab.index},
                       index=["lnb", "c"]).T

repr_floor = np.mean([emd(lab.loc[s, SIZE_COLS].to_numpy(), weibull_curve(*TARGETS.loc[s])) for s in lab.index])
print(f"best possible EMD with a Weibull curve (mean over training labels): {repr_floor:.2f}")
print(TARGETS.assign(type=lab["type"]).sort_values("lnb").round(3).T.to_string())

In [ ]:
# ---- head: standardise -> Ridge -> (ln b, c) per photo -> Weibull curve ------------------------
def build_matrix(rows_df: pd.DataFrame, feats, dino_branches, pcas=None):
    """Feature matrix = DoG columns (+ per-branch PCA of the raw DINOv2 embeddings when Patch B is active).

    Any PCA is fitted on the training rows only (no leakage); the fitted PCAs are returned for reuse."""
    parts = [rows_df[list(feats)].to_numpy(dtype=float)]
    fitted = {}
    for br in dino_branches:
        cols = [c for c in rows_df.columns if c.startswith(f"dino_{br}_")]
        D = rows_df[cols].to_numpy(dtype=float)
        if pcas is None:
            n = int(min(CFG["dino_pca_dim"], D.shape[0], D.shape[1]))
            p = PCA(n_components=n, svd_solver="full").fit(D)
            fitted[br] = p
        else:
            p = pcas[br]
        parts.append(p.transform(D))
    return np.hstack(parts), fitted


def fit_predict(train_rows: pd.DataFrame, pred_rows: pd.DataFrame, feats=None, dino_branches=(), per_image=False):
    """Standardise -> Ridge -> (ln b, c) per photo -> Weibull curve -> (optionally) average per sample."""
    feats = FEATS if feats is None else feats
    Xtr, pcas = build_matrix(train_rows, feats, dino_branches, pcas=None)
    Xte, _ = build_matrix(pred_rows, feats, dino_branches, pcas=pcas)
    scaler = StandardScaler().fit(Xtr)
    w = 1.0 / train_rows.groupby("sample_id").sample_id.transform("size")   # samples with more photos do not count more
    model = Ridge(alpha=CFG["ridge_alpha"]).fit(scaler.transform(Xtr),
                                                TARGETS.loc[train_rows.sample_id].to_numpy(),
                                                sample_weight=w.to_numpy())
    par = model.predict(scaler.transform(Xte))
    per_img = pd.DataFrame(weibull_curve(par[:, 0], par[:, 1]), columns=SIZE_COLS,
                           index=pred_rows.sample_id.to_numpy())
    return per_img if per_image else aggregate(per_img)

In [ ]:
# ---- smoke tests: the maths must be right before anything touches the LB ----------------------
rng = np.random.default_rng(0)

# 1) the Weibull fit recovers synthetic parameters exactly (the curve family contains itself)
F_syn = weibull_curve(np.log(3.0), 1.7)
lnb_hat, c_hat = fit_weibull_label(F_syn)
assert abs(np.exp(lnb_hat) - 3.0) < 0.05 and abs(c_hat - 1.7) < 0.05, (np.exp(lnb_hat), c_hat)

# 2) the metric: zero on the diagonal, symmetric, ~half-log-decade weights (EMD ~ 0.5 x sum|dF|)
assert float(emd(F_syn, F_syn)) == 0.0
_a, _b = rng.uniform(0, 100, 11), rng.uniform(0, 100, 11)
assert abs(float(emd(_a, _b)) - float(emd(_b, _a))) < 1e-9
assert np.allclose(W_EMD, 0.5, atol=0.01), W_EMD

# 3) both monotone projections produce valid rows; the L1-PAVA objective is never worse than running max
for _ in range(200):
    y = rng.uniform(0, 100, 11)
    p_rm = postprocess(y)
    assert p_rm.min() >= 0 and p_rm.max() <= 100 and (np.diff(p_rm) >= 0).all() and p_rm[-1] == 100.0
    obj = lambda x: float(np.abs(np.asarray(x) - y).sum())
    assert obj(l1_pava(y)) <= obj(np.maximum.accumulate(y)) + 1e-9, "L1-PAVA must be L1-optimal"

# 4) a curve through the real head stays valid (aggregation + postprocess on a fake per-photo table)
_demo = pd.DataFrame(np.tile(np.linspace(10, 100, 11), (3, 1)), index=["s1", "s1", "s2"], columns=SIZE_COLS)
assert (np.diff(aggregate(_demo).to_numpy(), axis=1) >= 0).all()

print("all smoke tests passed - EMD, Weibull fit, monotone projections, aggregation OK")

In [ ]:
# ---- the honest gate: site hold-out x phone swap (story report, section 7) --------------------
# Sites are grouped (sample IDs with the same letter and numbers within 12 -> 13 sites), each site is
# held out in turn, and within that the model trains on ONE phone and validates on the OTHER (both
# directions) - the only CV that imitates the train(Motorola/Samsung) -> test(iPhone) swap.
ids_sorted = sorted(lab.index, key=lambda s: (s[0], int(s[1:])))
SITE, _g, _prev = {}, 0, None
for _s in ids_sorted:                      # same letter and number gap <= 12 -> same site
    if _prev is None or _s[0] != _prev[0] or int(_s[1:]) - int(_prev[1:]) > 12:
        _g += 1
    SITE[_s] = f"S{_g:02d}"
    _prev = _s
SITE = pd.Series(SITE)


def weighted(per: dict) -> float:
    """0.2 x gravel mean + 0.8 x non-gravel mean (resembles the test composition)."""
    per = pd.Series(per)
    grp = lab.loc[per.index, "group"]
    return float(sum(CFG["weights_group"][k] * per[grp == k].mean() for k in ["gravel", "non_gravel"]))


tr = photos[photos.split == "train"]


def cross_phone_cv(feats=None, dino_branches=()):
    """Site hold-out x phone swap. Returns {sample_id: mean EMD over both directions}."""
    xcam = {}
    for st in sorted(SITE.unique()):
        hold = list(SITE.index[SITE.eq(st)])
        for src, dst in [(CAM_EDGE, CAM_A52), (CAM_A52, CAM_EDGE)]:
            v_rows = tr[tr.sample_id.isin(hold) & (tr.camera == dst)]
            if len(v_rows) == 0:
                continue
            p = fit_predict(tr[~tr.sample_id.isin(hold) & (tr.camera == src)], v_rows,
                            feats=feats, dino_branches=dino_branches)
            for s in p.index:
                xcam.setdefault(s, []).append(float(emd(lab.loc[s, SIZE_COLS].to_numpy(), p.loc[s].to_numpy())))
    return {s: float(np.mean(v)) for s, v in xcam.items() if len(v) == 2}


def site_loso(feats=None, dino_branches=()):
    """Site leave-one-out (in-domain reference; never used for selection)."""
    loso = {}
    for st in sorted(SITE.unique()):
        hold = list(SITE.index[SITE.eq(st)])
        p = fit_predict(tr[~tr.sample_id.isin(hold)], tr[tr.sample_id.isin(hold)],
                        feats=feats, dino_branches=dino_branches)
        for s in p.index:
            loso[s] = float(emd(lab.loc[s, SIZE_COLS].to_numpy(), p.loc[s].to_numpy()))
    return loso


xcam_A = cross_phone_cv()
loso_A = site_loso()
print(f"reference from the story report (5-feature model): cross-phone 50.67, LOSO 47.53")
print(f"CV site x cross-phone (weighted): {weighted(xcam_A):.2f}  ({len(xcam_A)} samples)")
print(f"CV site leave-one-out (weighted): {weighted(loso_A):.2f}  ({len(loso_A)} samples)")
print(pd.Series(loso_A).groupby(lab.loc[list(loso_A), "type"]).mean().round(1).rename("LOSO EMD by soil type").to_string())

In [ ]:
# ---- Patch B: frozen DINOv2 embeddings (dino_rgb + dino_gray) ---------------------------------
# Same philosophy as model A: no network is trained. DINOv2 is FROZEN and only summarises each photo
# into a vector; the head stays the tiny Ridge -> Weibull(ln b, c). The inputs go through the same
# train-like degradation as model A (so the embedding cannot tell phones apart through sharpness),
# square-resized to 518 px for the ViT. Gate decision happens in the next cell - never on the LB.
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

_DINO = {}


def get_dino():
    """Load the frozen DINOv2 backbone once (torch.hub, timm fallback). Returns None if unavailable."""
    global _DINO
    if "attempted" in _DINO:
        return _DINO["model"]
    _DINO["attempted"] = True
    if not (HAVE_TORCH and INTERNET):
        print("[Patch B] DINOv2 unavailable (no torch or no internet) - the gate runs on model A only")
        _DINO["model"] = None
        return None
    try:
        m = torch.hub.load("facebookresearch/dinov2", CFG["dino_model"])
    except Exception as e_hub:
        try:
            import timm
            m = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True)
        except Exception as e_timm:
            print("[Patch B] DINOv2 download failed:", e_hub, "|", e_timm)
            _DINO["model"] = None
            return None
    m.eval().to(DEVICE)
    _DINO["model"] = m
    return m


def load_dino_input(path: Path, ppm_eff: float) -> np.ndarray:
    """Train-like degradation (if sharp) -> landscape -> square resize to dino_px. uint8 RGB."""
    im = ImageOps.exif_transpose(Image.open(path))
    if ppm_eff > CFG["train_like_above_ppm"]:
        rgb, _ = to_train_like(np.asarray(im.convert("RGB")), ppm_eff)
        im = Image.fromarray(rgb)
    a = np.asarray(im.convert("RGB"))
    if a.shape[0] > a.shape[1]:
        a = np.ascontiguousarray(np.rot90(a))
    return cv2.resize(a, (CFG["dino_px"], CFG["dino_px"]), interpolation=cv2.INTER_AREA)


def dino_norm(a: np.ndarray) -> np.ndarray:
    return (a.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD


def dino_embed(model, img_norm: np.ndarray) -> np.ndarray:
    """Frozen forward pass -> [cls token, mean patch token] of the last layer."""
    with torch.no_grad():
        x = torch.from_numpy(img_norm).permute(2, 0, 1)[None].to(DEVICE)
        try:
            patch, cls = model.get_intermediate_layers(x, n=1, return_class_token=True)[0]
            emb = torch.cat([cls, patch.mean(1)], dim=1)
        except Exception:
            tok = model.forward_features(x)
            emb = torch.cat([tok[:, 0], tok[:, 1:].mean(1)], dim=1)
    return emb[0].float().cpu().numpy()


model_dino = get_dino()
if model_dino is not None:
    t0 = time.time()
    e_rgb, e_gray = [], []
    for r in photos.itertuples():
        a = load_dino_input(r.path, r.ppm_eff)
        e_rgb.append(dino_embed(model_dino, dino_norm(a)))
        g = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
        e_gray.append(dino_embed(model_dino, dino_norm(np.repeat(g[..., None], 3, axis=2))))
    e_rgb, e_gray = np.stack(e_rgb).astype(np.float32), np.stack(e_gray).astype(np.float32)
    dino_df = pd.DataFrame(np.hstack([e_rgb, e_gray]),
                           columns=[f"dino_rgb_{i:03d}" for i in range(e_rgb.shape[1])]
                                   + [f"dino_gray_{i:03d}" for i in range(e_gray.shape[1])],
                           index=photos.index)
    photos = photos.join(dino_df)
    print(f"DINOv2 embeddings for {len(photos)} photos in {time.time() - t0:.0f} s "
          f"({e_rgb.shape[1]}+{e_gray.shape[1]} dims per photo)")
    # diagnostic: do the embeddings still cluster by phone (bad) or by soil (good)?
    _pca2 = PCA(n_components=2, svd_solver="full")
    _s = photos[photos.split == "train"].groupby("sample_id")[[c for c in dino_df.columns if c.startswith("dino_rgb_")]].mean()
    _z = _pca2.fit_transform(_s.to_numpy())
    _fig, _ax = plt.subplots(figsize=(5.5, 4.2), constrained_layout=True)
    _cams = photos[photos.split == "train"].groupby("sample_id").camera.first()
    for _cam in sorted(_cams.unique()):
        _m = (_cams.loc[_s.index] == _cam).to_numpy()
        _ax.scatter(_z[_m, 0], _z[_m, 1], s=28, label=_cam, alpha=0.85)
    _ax.set_title("DINOv2 (rgb) per training sample, PCA 2D - phone clustering would be bad news")
    _ax.set_xlabel("PC1"); _ax.set_ylabel("PC2"); _ax.legend(fontsize=8); _ax.grid(alpha=0.3)
    plt.show()
else:
    print("[Patch B] skipped - continuing with model A features only")

In [ ]:
# ---- the Patch B gate: DINOv2 is switched on ONLY if cross-phone CV clearly improves -----------
# Lesson from the story report (the 59.59 regression): never select on in-domain CV or LOSO, never
# bundle changes. One gate, one margin, parsimony on ties. FEATS (the 5 DoG features) always stay -
# DINOv2 can only be ADDED, mirroring 'dog5 concat' in the plan.
HAVE_DINO = model_dino is not None
GATE_CFGS = [("A   dog5 (baseline)", ())]
if HAVE_DINO:
    GATE_CFGS += [("B1  dog5 + dino_rgb", ("rgb",)),
                  ("B2  dog5 + dino_gray", ("gray",)),
                  ("B3  dog5 + dino_rgb + dino_gray", ("rgb", "gray"))]

gate_rows, xcam_cfg = [], {}
for _name, _br in GATE_CFGS:
    _xc = cross_phone_cv(dino_branches=_br)
    xcam_cfg[_name] = _xc
    gate_rows.append({"config": _name, "branches": _br,
                      "cross_phone_weighted": weighted(_xc), "n_samples": len(_xc)})
gate = pd.DataFrame(gate_rows)
print(gate.drop(columns="branches").round(2).to_string(index=False))

_base = gate.loc[0, "cross_phone_weighted"]
_best_i = int(gate.cross_phone_weighted.idxmin())
_best = gate.loc[_best_i]
if _best.branches and (_base - _best.cross_phone_weighted) >= CFG["gate_margin"]:
    PATCH_B_BRANCHES, _why = _best.branches, (
        f"{_best.config} wins by {_base - _best.cross_phone_weighted:.2f} >= margin {CFG['gate_margin']:.1f}")
else:
    PATCH_B_BRANCHES, _why = (), (
        f"kept model A (best DINO gain {np.nan if not HAVE_DINO else _base - gate.cross_phone_weighted[1:].min():.2f} "
        f"< margin {CFG['gate_margin']:.1f}, or no DINO available) - parsimony rule of the story report")
print(f"PATCH B DECISION: branches={PATCH_B_BRANCHES} -> {_why}")

In [ ]:
# ---- final model A: fit on all training photos, predict the test set ---------------------------
te = photos[photos.split == "test"]
pred_img = fit_predict(tr, te, dino_branches=PATCH_B_BRANCHES, per_image=True)
pred_A = aggregate(pred_img)
print(pred_A.round(1).to_string())

In [ ]:
# ---- step B: measuring stones with SAM (story report, sections 9-10) ---------------------------
# Model A cannot say "30 % of this sample is > 63 mm": the 24 training curves almost never contain
# cobbles. But the stones are visible in the full-resolution test photos, so SAM is used to MEASURE
# them (never to predict curves). Only photos with ppm_eff >= 12 px/mm are measured - the test photos
# and the 3 full-res photos of training sample H374 (the only measurable training sample).
SAM_OK = HAVE_TORCH and HAVE_TRANSFORMERS and INTERNET
if SAM_OK:
    from transformers import pipeline as hf_pipeline
    try:
        mask_gen = hf_pipeline("mask-generation", model=CFG["sam_model"],
                               device=0 if DEVICE == "cuda" else -1)
        print("SAM on", "GPU" if DEVICE == "cuda" else "CPU (slow: ~30 s per photo)")
    except Exception as e:
        print("[stones] SAM could not load -> submission degrades to model A only:", e)
        SAM_OK = False


def load_work(path: Path, ppm: float) -> np.ndarray:
    """EXIF rotation -> 2 px/mm -> landscape -> cut the borders. RGB uint8.

    At 2 px/mm stone outlines survive but the surface pattern that makes segmenters split stones is
    already blurred away."""
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    w, h = im.size
    f = CFG["work_ppm"] / ppm
    a = cv2.resize(np.asarray(im), (round(w * f), round(h * f)), interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]:
        a = np.ascontiguousarray(np.rot90(a))
    H, W = a.shape[:2]
    c = CFG["work_crop"]
    return a[int(H * c):int(H * (1 - c)), int(W * c):int(W * (1 - c))]


def stones_from_masks(masks, tot: int):
    """Keep stone-like, non-overlapping masks (visited large -> small).

    Kept if: area >= (8 mm)^2, <= 50 % of the image, not more than half covered by an already accepted
    stone, and fill >= 35 % of the circle on the long axis (4 sigma of the pixel coordinates, in mm).
    Returns the stones, the covered fraction, and a label image for plotting."""
    order = sorted(range(len(masks)), key=lambda i: -int(masks[i].sum()))
    taken = np.zeros_like(masks[0], dtype=bool)
    label_img = np.zeros(masks[0].shape, dtype=np.int32)
    stones = []
    for i in order:
        m = masks[i]
        sz = int(m.sum())
        if sz < (CFG["min_side_mm"] * CFG["work_ppm"]) ** 2 or sz > CFG["max_area_frac"] * tot:
            continue
        if (m & taken).sum() > CFG["overlap_drop"] * sz:
            continue
        ys, xs = np.nonzero(m)
        pts = np.stack([xs, ys], 1).astype(np.float32)
        pts -= pts.mean(0)
        major = 4.0 * float(np.sqrt(max(np.linalg.eigvalsh(np.cov(pts.T)).max(), 1e-9))) / CFG["work_ppm"]
        fill = sz / (np.pi * (major * CFG["work_ppm"] / 2) ** 2 + 1e-9)
        if fill < CFG["min_fill"]:
            continue
        taken |= m
        stones.append({"area_frac": sz / tot, "major_mm": major})
        label_img[m & (label_img == 0)] = len(stones)
    return stones, float(taken.mean()), label_img


keep_for_plot, meas = {}, []
if SAM_OK:
    targets = photos[photos.ppm_eff >= CFG["sam_min_ppm"]]
    print(f"measuring {len(targets)} photos:", targets.groupby(["split", "sample_id"]).size().to_dict())
    t0 = time.time()
    for r in targets.itertuples():
        rgb = load_work(r.path, r.ppm_eff)
        out = mask_gen(Image.fromarray(rgb), points_per_batch=CFG["points_per_batch"],
                       pred_iou_thresh=CFG["pred_iou_thresh"], stability_score_thresh=CFG["stability_score_thresh"])
        stones, cover, label_img = stones_from_masks([np.asarray(m) for m in out["masks"]], rgb.shape[0] * rgb.shape[1])
        meas.append({"file": r.path.name, "split": r.split, "sample_id": r.sample_id, "cover": cover,
                     "n_stones": len(stones),
                     "A63": sum(s["area_frac"] for s in stones if s["major_mm"] > 63),
                     "A20": sum(s["area_frac"] for s in stones if s["major_mm"] > 20),
                     "max_stone_mm": max((s["major_mm"] for s in stones), default=0.0)})
        if r.sample_id not in keep_for_plot:
            keep_for_plot[r.sample_id] = (rgb, label_img)
    meas = pd.DataFrame(meas)
    print(f"done in {time.time() - t0:.0f} s")
else:
    meas = pd.DataFrame(columns=["file", "split", "sample_id", "cover", "n_stones", "A63", "A20", "max_stone_mm"])
    print("[stones] SAM unavailable - every sample stays model A (the submission is still valid)")

if SAM_OK and len(meas):
    ms = meas.groupby("sample_id").agg(photos=("file", "size"), A63=("A63", "mean"), A20=("A20", "mean"),
                                       cover=("cover", "mean"), max_stone_mm=("max_stone_mm", "max"))
    print(ms.round(3).sort_values("A20", ascending=False).to_string())
    # visual check (each colour = one accepted stone)
    _rng = np.random.default_rng(0)
    show = [s for s in ["HPC_Muenster_BS6_9_0-10m", "HPC_Testfeld Lidl WHV", "H374", "HPC_Airbus BS6-3"] if s in keep_for_plot]
    _fig, _axes = plt.subplots(2, max(len(show), 1), figsize=(4.5 * max(len(show), 1), 6.5), constrained_layout=True)
    for _j, _sid in enumerate(show):
        _rgb, _lab = keep_for_plot[_sid]
        _col = _rng.integers(40, 255, size=(_lab.max() + 1, 3)).astype(np.uint8)
        _col[0] = 0
        _over = np.where(_lab[..., None] > 0, (0.45 * _rgb + 0.55 * _col[_lab]).astype(np.uint8), _rgb)
        _axes[0, _j].imshow(_rgb); _axes[0, _j].set_title(_sid[:28], fontsize=9)
        _axes[1, _j].imshow(_over)
        _m = ms.loc[_sid]
        _axes[1, _j].set_title(f"A63 {_m.A63:.2f}  A20 {_m.A20:.2f}  cover {_m.cover:.2f}", fontsize=9)
        for _a in _axes[:, _j]:
            _a.axis("off")
    plt.show()

In [ ]:
# ---- Patch A: from stone area to the 63 / 20 mm points, three k-modes --------------------------
# Area on the surface is not sieved mass, so ONE factor k converts area share to mass share:
#   20 mm point = 100 - k20 * 100 * A20,  63 mm point = 100 - k63 * 100 * A63  (>= the 20 mm point),
# and every point <= 20 mm is scaled by (new 20 / old 20) when the curve moves to the coarse side.
#
# k-modes (each is one submission - never bundle):
#   "delesse" : k20 = k63 = 1.0   - Patch A. Delesse's stereological principle says the area share of
#                                  stones > d on a random surface section estimates the volume share,
#                                  i.e. k = 1 by theory. The H374 calibration (v4) was biased high
#                                  because SAM covers only ~30 % of H374's darker, softer photos.
#   "v3"      : k20 = 1.00, k63 = 0.92 - reverse-engineered from the 34.32 submission.
#   "v4"      : k20 = k63 = K_CAL - the single H374-calibrated factor of the 36.19 submission.
I20, I63 = SIZE_COLS.index("20"), SIZE_COLS.index("63")


def redraw(curve: np.ndarray, a63: float, a20: float, k20: float, k63: float) -> np.ndarray:
    """Replace the 63 / 20 mm points by the measurement; scale the points <= 20 mm when moving coarse."""
    c = np.asarray(curve, dtype=float).copy()
    new20 = max(100.0 - k20 * 100.0 * a20, 0.0)
    new63 = max(100.0 - k63 * 100.0 * a63, new20)
    scale = new20 / max(c[I20], 1e-6)
    if scale < 1:
        c[: I20 + 1] *= scale
    c[I20], c[I63], c[-1] = new20, new63, 100.0
    return c


K_MODES = {}
K_CAL = None
if SAM_OK and len(meas) and "H374" in ms.index:
    K_CAL = (100.0 - lab.loc["H374", "20"]) / (100.0 * ms.loc["H374", "A20"])
    K_MODES = {"delesse": (1.0, 1.0), "v3": (1.00, 0.92), "v4": (K_CAL, K_CAL)}
    print(f"K_CAL (v4, from H374) = {K_CAL:.3f}   "
          f"(true >20 mm {100 - lab.loc['H374', '20']:.1f} % vs measured A20 {100 * ms.loc['H374', 'A20']:.1f} %)")
    print("reference from the story report (CPU run): k ~ 1.24, Muenster A63 ~ 0.25, A20 ~ 0.72")

    # the only training check that exists for step B: H374 held out, redrawn with each k-mode
    h374_A = fit_predict(tr[tr.sample_id != "H374"], tr[tr.sample_id == "H374"]).loc["H374"]
    print("\nH374 held-out check (lower EMD is better):")
    for _km, (_k20, _k63) in K_MODES.items():
        _e = float(emd(lab.loc["H374", SIZE_COLS].to_numpy(),
                       postprocess(redraw(h374_A.to_numpy(), ms.loc["H374", "A63"], ms.loc["H374", "A20"], _k20, _k63))))
        print(f"  k_mode={_km:8s} k20={_k20:.3f} k63={_k63:.3f} -> EMD {_e:.2f}")
    print(f"  (model A without stones: EMD {float(emd(lab.loc['H374', SIZE_COLS].to_numpy(), h374_A.to_numpy())):.2f})")

# apply the selected k-mode (samples with A63 >= 2 % are touched; in practice only Muenster)
if SAM_OK and len(meas) and K_MODES:
    K20, K63 = K_MODES[CFG["k_mode"]]
    final, changed = {}, []
    for sid in pred_A.index:
        a63 = float(ms.loc[sid, "A63"]) if sid in ms.index else 0.0
        a20 = float(ms.loc[sid, "A20"]) if sid in ms.index else 0.0
        if a63 >= CFG["apply_if_A63_ge"]:
            final[sid] = redraw(pred_A.loc[sid].to_numpy(), a63, a20, K20, K63)
            changed.append(sid)
        else:
            final[sid] = pred_A.loc[sid].to_numpy()
    pred_B = pd.DataFrame(postprocess(np.stack([final[s] for s in pred_A.index])),
                          index=pred_A.index, columns=SIZE_COLS)
    print(f"\nselected k_mode = '{CFG['k_mode']}' -> k20={K20:.3f}, k63={K63:.3f}; redrawn samples: {changed}")
    for sid in changed:
        print(pd.DataFrame({"model A": pred_A.loc[sid], "A + stones": pred_B.loc[sid]}).T.round(1).to_string(), "\n")
else:
    pred_B = pred_A.copy()
    changed = []
    print("no redraw - pred_B = pred_A")

# curves: H374 calibration + every redrawn sample
_show = (["H374"] if (SAM_OK and len(meas) and "H374" in ms.index) else []) + changed
if _show:
    _fig, _axes = plt.subplots(1, len(_show), figsize=(6 * len(_show), 4), constrained_layout=True, squeeze=False)
    for _ax, _sid in zip(_axes[0], _show):
        _ax.semilogx(SIZES, lab.loc["H374", SIZE_COLS] if _sid == "H374" else pred_A.loc[_sid], "C0--o", label="model A")
        if _sid == "H374":
            _k20, _k63 = K_MODES[CFG["k_mode"]]
            _ax.semilogx(SIZES, postprocess(redraw(h374_A.to_numpy(), ms.loc["H374", "A63"], ms.loc["H374", "A20"], _k20, _k63)),
                         "C2-o", label="A + stones (held out)")
            _ax.semilogx(SIZES, lab.loc["H374", SIZE_COLS], "k-o", label="truth")
        else:
            _ax.semilogx(SIZES, pred_B.loc[_sid], "C3-o", label="A + stones")
        _ax.set_title(_sid); _ax.set_xlabel("grain size [mm]"); _ax.set_ylabel("% passing")
        _ax.grid(alpha=0.3, which="both"); _ax.legend(fontsize=8)
    plt.show()

In [ ]:
# ---- submission (plus one-change-per-submission variants) --------------------------------------
def write_sub(curves: pd.DataFrame, name: str) -> pd.DataFrame:
    sub = sample_sub[["sample_id"]].merge(curves, left_on="sample_id", right_index=True, how="left")
    vals = sub[SIZE_COLS].to_numpy()
    assert list(sub.columns) == list(sample_sub.columns), "column mismatch"
    assert not np.isnan(vals).any(), "missing prediction"
    assert ((vals >= 0) & (vals <= 100)).all() and (np.diff(vals, axis=1) >= 0).all() and (vals[:, -1] == 100).all()
    sub.to_csv(OUTPUT_DIR / name, index=False, float_format="%.4f")
    return sub


write_sub(pred_B, "submission.csv")
write_sub(pred_A, "submission_model_A_only.csv")
if SAM_OK and len(meas) and K_MODES:
    for _km, (_k20, _k63) in K_MODES.items():
        if _km == CFG["k_mode"]:
            continue
        _alt, _ = {}, None
        for sid in pred_A.index:
            _a63 = float(ms.loc[sid, "A63"]) if sid in ms.index else 0.0
            _a20 = float(ms.loc[sid, "A20"]) if sid in ms.index else 0.0
            _alt[sid] = (redraw(pred_A.loc[sid].to_numpy(), _a63, _a20, _k20, _k63)
                         if _a63 >= CFG["apply_if_A63_ge"] else pred_A.loc[sid].to_numpy())
        write_sub(pd.DataFrame(postprocess(np.stack([_alt[s] for s in pred_A.index])),
                               index=pred_A.index, columns=SIZE_COLS), f"submission_k_{_km}.csv")
print("wrote:", sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))
print(f"selected configuration: dino_branches={PATCH_B_BRANCHES}, k_mode='{CFG['k_mode']}' (K20={K20 if K_MODES else 'n/a'})")

## Closing notes — read before submitting (story report, sections 11–12)

**What this notebook changed relative to the 36.19 / 34.32 submissions**

1. **Patch A (always part of the main output):** `k_mode = "delesse"` replaces the H374-calibrated factor with
   the stereological identity "area share = volume share" (Delesse, 1848). The H374 fit was biased high
   because SAM covers only ~30 % of H374's darker photos. Expected Public LB ≈ 34.6 (between the 34.32 of the
   per-point k and the 36.19 of k ≈ 1.22 — the theory says the honest number is k = 1).
2. **Patch B (gated, never assumed):** frozen DINOv2 embeddings enter the head **only** if site × cross-phone CV
   improves by at least 1.0 EMD over the 5-feature baseline (50.67 reference). If the gate rejects, nothing changes —
   that is the parsimony rule that would have prevented the 59.59 regression.
3. **The stones section is unchanged** from the v1 submission except that k becomes a per-point pair
   (k20, k63) so the three k-modes are exactly one change each.

**Submission plan (one change at a time, in this order)**

| # | file | delta vs previous | what it tests |
|---|---|---|---|
| 1 | `submission.csv` (k = delesse) | baseline + Patch A | the stereological identity |
| 2 | best of `submission_k_v3.csv` / `submission_k_v4.csv` | only if #1 moves the wrong way | k as pure LB calibration |
| 3 | DINO variant (re-run with `CFG["gate_margin"] = 0`) | only if the gate was close | frozen embeddings, same head |

**Honest caveats** (unchanged from the story report): the LB gain of step B comes from one sample
(`HPC_Muenster_BS6_9_0-10m`); k rests on a single training sample with 3 photos; area is not mass and a
surface is not a volume — the single factor absorbs this only on average; SAM on GPU vs CPU can shift
A63/A20 slightly, so the redrawn row may differ a little between runs. The private set is iPhone-16-heavy
(7 of 10 samples), and iPhone-16 photos show no stone above ~20 mm — the texture model A carries ~90 % of
the final score, which is exactly why Patch B is gated on the cross-phone gate instead of the public LB.